# 📘 SQL para Data Engineering
### Guía personal de consultas — Proyecto Olist

> **Cómo usar este notebook:**
> - Las celdas **grises** son código ejecutable. Pulsa `Shift + Enter` para ejecutar.
> - Las celdas **blancas** son notas y explicaciones.
> - Añade tus propias notas donde veas 📝

---

## ⚙️ Setup — Ejecutar siempre al abrir el notebook

Esto conecta Jupyter con PostgreSQL. **Obligatorio ejecutar antes de cualquier query.**

Primero instala jupysql si no lo tienes:
```bash
pip install jupysql psycopg2-binary
```

In [24]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Cambia 'tu_usuario' por tu usuario de Mac (comando: whoami en terminal)
%sql postgresql://tomas@localhost:5432/data_engineering

print("✅ Conexión establecida con PostgreSQL")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
✅ Conexión establecida con PostgreSQL


---
## 1. Exploración básica de tablas

Antes de escribir queries complejas, siempre explora la tabla para entender qué datos tienes.

### 📌 Ver las tablas disponibles en la base de datos

En psql (terminal) se hace con `\dt`.
Desde Jupyter hacemos una query al catálogo interno de PostgreSQL.

In [3]:
%%sql
-- Ver todas las tablas de la base de datos
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name;

,table_name
0,orders


### 📌 Ver las columnas de una tabla

Equivalente a `\d orders` en psql.
Útil para recordar los nombres exactos de las columnas antes de escribir una query.

In [1]:
%%sql
-- Ver columnas y tipos de la tabla orders
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'orders'
ORDER BY ordinal_position;*

UsageError: Cell magic `%%sql` not found.


### 📌 Ver las primeras filas

Siempre lo primero cuando ves una tabla nueva.
`LIMIT` evita que te devuelva millones de filas de golpe.

In [4]:
%%sql
-- Ver las primeras 5 filas de la tabla
SELECT * FROM orders LIMIT 5;

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0


### 📌 Contar el total de filas

`COUNT(*)` cuenta todas las filas incluyendo nulos.
`COUNT(columna)` cuenta solo las filas donde esa columna **NO** es nula.

📝 **Mis notas:**

In [ ]:
%%sql
-- Contar el total de pedidos
SELECT COUNT(*) as total_pedidos FROM orders;

---
## 2. Filtrar filas — WHERE

`WHERE` filtra las filas que cumplen una condición.
Equivalente a `orders[condición]` en Pandas.

```sql
SELECT * FROM tabla WHERE condición;
```

**Operadores más usados:**

| Operador | Significado | Ejemplo |
|----------|-------------|------|
| `=` | igual | `order_status = 'delivered'` |
| `!=` | distinto | `order_status != 'canceled'` |
| `>` `<` | mayor / menor | `delivery_days > 30` |
| `>=` `<=` | mayor o igual / menor o igual | `delivery_days >= 7` |
| `IS NULL` | es nulo | `order_approved_at IS NULL` |
| `IS NOT NULL` | no es nulo | `order_approved_at IS NOT NULL` |
| `IN` | está en una lista | `order_status IN ('delivered', 'shipped')` |
| `BETWEEN` | entre dos valores | `delivery_days BETWEEN 1 AND 7` |

📝 **Mis notas:**

In [5]:
%%sql
-- Pedidos que tardaron más de 30 días
SELECT COUNT(*) as pedidos_lentos
FROM orders
WHERE delivery_days > 30;

,pedidos_lentos
0,4117


In [6]:
%%sql
-- Pedidos entregados el mismo día (0 días)
SELECT *
FROM orders
WHERE delivery_days = 0
LIMIT 10;

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days
0,38c1e3d4ed6a13cd0cf612d4c09766e9,18c934f4cdc994cd04eb13bce3f47a18,delivered,2018-02-02 15:26:38,2018-02-02 16:00:16,2018-02-03 01:18:26,2018-02-03 15:05:56,2018-02-20,0.0
1,d3ca7b82c922817b06e5ca21165c5ea2,d23df2c6c3e51d875f458d123b2b3c90,delivered,2017-11-16 13:54:08,2017-11-16 14:08:23,2017-11-16 20:56:53,2017-11-17 13:49:40,2017-11-29,0.0
2,1d893dd7ca5f77ebf5f59f0d2017eee0,b19da0df0271e8a3553e3670f86aeab5,delivered,2017-06-19 08:19:45,2017-06-19 08:30:20,2017-06-19 13:32:04,2017-06-19 21:07:52,2017-06-30,0.0
3,21a8ffca665bc7a1087d31751a7b7cbc,225aed9e773953084b09cf496c2be05a,delivered,2017-05-31 12:00:35,2017-05-31 13:07:28,2017-05-31 12:43:47,2017-06-01 10:28:24,2017-06-13,0.0
4,f3c6775ba3d2d9fe2826f93b71f12008,6aef84c09844a371d82a49152c550b95,delivered,2017-07-04 11:37:47,2017-07-04 11:50:21,2017-07-04 13:53:13,2017-07-05 08:09:26,2017-07-17,0.0
5,434cecee7d1a65fc65358a632b6f725f,922a46283625e9c096bfd998913c470c,delivered,2017-05-29 13:21:46,2017-05-29 13:30:24,2017-05-29 14:54:51,2017-05-30 08:06:56,2017-06-19,0.0
6,f349cdb62f69c3fae5c4d7d3f3a4a185,c5e200d485ae35a7036cc2e7c1d8ea81,delivered,2018-06-28 14:34:48,2018-06-28 14:50:48,2018-06-28 18:08:00,2018-06-29 14:12:18,2018-07-12,0.0
7,d5fbeedc85190ba88580d6f82d1d5ed3,344423c2e26d47d2b6d3dd363a89e812,delivered,2017-05-15 11:50:53,2017-05-15 12:12:07,2017-05-15 12:52:34,2017-05-16 10:21:52,2017-05-24,0.0
8,e65f1eeee1f52024ad1dcd03447f7482,198f511b5a75bf936a96f1d4769e3974,delivered,2018-05-18 15:03:19,2018-05-18 15:15:37,2018-05-18 15:00:00,2018-05-19 12:28:30,2018-05-29,0.0
9,79e324907160caea526fd8b94389dbbc,331d79b67223ee7e5cd31d3e03e4cfcc,delivered,2018-06-18 12:59:42,2018-06-18 13:16:46,2018-06-18 14:52:00,2018-06-19 12:43:27,2018-06-28,0.0


In [ ]:
%%sql
-- Pedidos sin fecha de aprobación (nulos)
SELECT COUNT(*) as sin_aprobar
FROM orders
WHERE order_approved_at IS NULL;

---
## 3. Agrupar y contar — GROUP BY

`GROUP BY` agrupa las filas con el mismo valor y permite aplicar funciones de agregación sobre cada grupo.

```sql
SELECT columna, FUNCIÓN(*)
FROM tabla
GROUP BY columna;
```

**Funciones de agregación más usadas:**

| Función | Qué hace |
|---------|----------|
| `COUNT(*)` | cuenta filas |
| `SUM(col)` | suma valores |
| `AVG(col)` | media |
| `MAX(col)` | máximo |
| `MIN(col)` | mínimo |

⚠️ **Regla importante:** toda columna en el `SELECT` que no sea una función de agregación, debe estar en el `GROUP BY`.

📝 **Mis notas:**

⚠️ NULLS en ORDER BY — comportamiento importante
Por defecto en PostgreSQL:
ORDER BY col DESC → NULLs aparecen PRIMERO (como si fueran el valor más grande)
ORDER BY col ASC  → NULLs aparecen ÚLTIMO

Para controlarlo explícitamente:
ORDER BY col DESC NULLS LAST  → NULLs al final
ORDER BY col DESC NULLS FIRST → NULLs al principio
ORDER BY col ASC NULLS LAST   → NULLs al final
ORDER BY col ASC NULLS FIRST  → NULLs al principio

In [ ]:
%%sql
-- Contar pedidos por estado
SELECT order_status, COUNT(*) as total
FROM orders
GROUP BY order_status
ORDER BY total DESC;

---
## 4. Funciones de agregación — AVG, MAX, MIN, SUM

⚠️ **Nota PostgreSQL:** `ROUND` no acepta `double precision` directamente.
Hay que convertir con `::numeric`.

```sql
ROUND(AVG(columna)::numeric, 2)
```

📝 **Mis notas:**

In [ ]:
%%sql
-- Media de días de entrega
-- ::numeric es necesario en PostgreSQL para usar ROUND con double precision
SELECT ROUND(AVG(delivery_days)::numeric, 2) as media_dias
FROM orders;

In [ ]:
%%sql
-- Máximo y mínimo de días de entrega
SELECT 
    MAX(delivery_days) as maximo_dias,
    MIN(delivery_days) as minimo_dias
FROM orders;

In [ ]:
%%sql
-- Estadísticas completas en una sola query
SELECT
    COUNT(*)                                    as total_pedidos,
    ROUND(AVG(delivery_days)::numeric, 2)       as media_dias,
    MAX(delivery_days)                          as maximo_dias,
    MIN(delivery_days)                          as minimo_dias
FROM orders;

---
## 5. Ordenar resultados — ORDER BY

```sql
SELECT * FROM tabla ORDER BY columna ASC;   -- ascendente (por defecto)
SELECT * FROM tabla ORDER BY columna DESC;  -- descendente
```

📝 **Mis notas:**

In [ ]:
%%sql
-- Top 10 pedidos más lentos
SELECT order_id, delivery_days
FROM orders
ORDER BY delivery_days DESC
LIMIT 10;

---
## 6. Combinar condiciones — AND, OR

```sql
WHERE condición1 AND condición2   -- ambas deben cumplirse
WHERE condición1 OR condición2    -- basta con que una se cumpla
```

📝 **Mis notas:**

In [ ]:
%%sql
-- Pedidos que tardaron más de 30 días Y tienen fecha de entrega
SELECT COUNT(*) as pedidos_lentos_con_fecha
FROM orders
WHERE delivery_days > 30
AND order_delivered_customer_date IS NOT NULL;

---
## 7. Próximos temas — JOINs, CTEs, Window Functions

Estos temas los iremos añadiendo a medida que avancemos.

- [ ] **JOINs** — unir varias tablas
- [ ] **CTEs** — Common Table Expressions
- [ ] **Window Functions** — ROW_NUMBER, RANK, LAG, LEAD
- [ ] **Subqueries** — queries dentro de queries
- [ ] **HAVING** — filtrar después de GROUP BY

---
## 🏋️ Espacio para practicar

Escribe aquí tus propias queries y anótalas.

**Ideas:**
- ¿Cuántos pedidos tardaron entre 7 y 14 días?
- ¿Cuál es la media de días solo para pedidos que tardaron más de 1 día?
- ¿Cuántos pedidos tienen `order_approved_at` nulo?

📝 **Mis queries:**

In [10]:
%%sql
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name;


,column_name,data_type
0,order_id,text
1,customer_id,text
2,order_status,text
3,order_purchase_timestamp,timestamp without time zone
4,order_approved_at,timestamp without time zone
5,order_delivered_carrier_date,timestamp without time zone
6,order_delivered_customer_date,timestamp without time zone
7,order_estimated_delivery_date,timestamp without time zone
8,delivery_days,double precision


In [22]:
%%sql
select count(*) from orders where delivery_days between 7 and 14


,count
0,44048


In [23]:
%%sql
select ROUND(AVG(delivery_days)::numeric, 2) as media_dias from orders where delivery_days > 1


,media_dias
0,12.28


In [21]:
%%sql
select count(*) from orders where order_approved_at IS NULL


,count
0,14


## 8. Window Functions — LAG y LEAD

Permiten acceder al valor de la fila anterior o siguiente sin perder el detalle de cada fila.

- `LAG(col)` → valor de la fila **anterior**
- `LEAD(col)` → valor de la fila **siguiente**

Siempre van con `OVER (ORDER BY col)` para definir el orden.

**Cuándo usarlas:** comparar un valor con el período anterior.
Por ejemplo: crecimiento mes a mes, diferencia respecto al día anterior, etc.

⚠️ La primera fila siempre tendrá `NULL` en LAG (no hay fila anterior).
⚠️ La última fila siempre tendrá `NULL` en LEAD (no hay fila siguiente).

**DATE_TRUNC:** recorta una fecha a un nivel de precisión.
- `DATE_TRUNC('month', fecha)` → agrupa por mes
- `DATE_TRUNC('year', fecha)`  → agrupa por año
- `DATE_TRUNC('day', fecha)`   → agrupa por día

In [ ]:
%%sql
-- Evolución mensual con comparación al mes anterior
WITH pedidos_por_mes AS (
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp) AS mes,
        COUNT(*) AS total_pedidos
    FROM orders
    GROUP BY mes
    ORDER BY mes
)
SELECT
    mes,
    total_pedidos,
    LAG(total_pedidos)  OVER (ORDER BY mes) AS pedidos_mes_anterior,
    LEAD(total_pedidos) OVER (ORDER BY mes) AS pedidos_mes_siguiente,
    total_pedidos - LAG(total_pedidos) OVER (ORDER BY mes) AS diferencia
FROM pedidos_por_mes;

In [5]:
## 9. Window Functions — ROW_NUMBER vs RANK vs DENSE_RANK

Las tres asignan un número a cada fila, pero tratan los empates diferente.

| Función | Empates | Huecos |
|---------|---------|--------|
| `ROW_NUMBER` | Número único siempre | No aplica |
| `RANK` | Mismo número | Sí deja huecos |
| `DENSE_RANK` | Mismo número | No deja huecos |

**Cuándo usar cada una:**
- `ROW_NUMBER` → necesitas un ID único por fila
- `RANK` → competiciones oficiales (1, 2, 2, 4...)
- `DENSE_RANK` → análisis de datos (1, 2, 2, 3...)

⚠️ **No puedes usar alias del SELECT en el WHERE de la misma query.**
Si necesitas filtrar por un alias, mete la query en una CTE primero.

📝 **Mis notas:**

SyntaxError: invalid syntax (2584405629.py, line 3)

In [ ]:
%%sql
-- Comparación de ROW_NUMBER, RANK y DENSE_RANK
-- Fíjate en las filas donde delivery_days es el mismo valor
SELECT
    order_id,
    delivery_days,
    ROW_NUMBER()  OVER (ORDER BY delivery_days ASC NULLS LAST) AS row_number,
    RANK()        OVER (ORDER BY delivery_days ASC NULLS LAST) AS rank,
    DENSE_RANK()  OVER (ORDER BY delivery_days ASC NULLS LAST) AS dense_rank
FROM orders
LIMIT 20;

In [4]:
## 10. CASE WHEN — Condicionales en SQL

Es el if/else de SQL. Crea una columna nueva con valores distintos según condiciones.

```sql
CASE
    WHEN condición1 THEN valor1
    WHEN condición2 THEN valor2
    ELSE valor_por_defecto
END AS nombre_columna
```

**Reglas importantes:**
- Las condiciones se evalúan en orden, la primera que se cumple gana
- `ELSE` es opcional pero recomendable para capturar casos no previstos
- Se puede usar en SELECT, WHERE, ORDER BY y GROUP BY
- En GROUP BY sí puedes usar el alias definido en SELECT (a diferencia de WHERE)

📝 **Mis notas:**

SyntaxError: invalid syntax (3794441162.py, line 3)

In [ ]:
%%sql
-- Clasificar pedidos por velocidad de entrega y contar cada categoría
SELECT
    CASE
        WHEN delivery_days < 7              THEN 'rapido'
        WHEN delivery_days BETWEEN 7 AND 20 THEN 'normal'
        WHEN delivery_days > 20             THEN 'lento'
        ELSE 'sin datos'
    END AS rango_entrega,
    COUNT(*) AS total_pedidos
FROM orders
GROUP BY rango_entrega
ORDER BY total_pedidos DESC;

In [ ]:
## 11. Subqueries — Queries dentro de queries

Una subquery es una query anidada dentro de otra, directamente entre paréntesis.

**Diferencia con CTE:**
- CTE → tiene nombre, más legible, mejor para queries complejas
- Subquery → anónima, más compacta, útil para queries simples

**Dónde puede ir una subquery:**
- En el `FROM` → como si fuera una tabla
- En el `WHERE` → para comparar con un valor calculado
- En el `SELECT` → para calcular un valor por fila

📝 **Mis notas:**

In [ ]:
%%sql
-- Pedidos que tardaron más que la media global
-- La subquery calcula la media, la query principal filtra
SELECT order_id, delivery_days
FROM orders
WHERE delivery_days > (
    SELECT AVG(delivery_days)
    FROM orders
)
ORDER BY delivery_days ASC;